03 — RF-DETR + SAM2 V1 tracking (reconstructed)
Ricostruzione della pipeline V1 che aveva prodotto il CSV poi visualizzato come
v_00HRwkvvjtQ_c001_tracking_v1_20ids.mp4.
Configurazione ripristinata:
- RF-DETR Nano fine-tuned
- SAM2 video predictor
- chunk = 100 frame
- overlap = 10 frame
- detection threshold = 0.30
- associazione tra chunk mediante IoU sul frame di continuità
- output: experiments/sam2_tracking/v_00HRwkvvjtQ_c001_tracks_RECONSTRUCTED.csv
Nota: 20 IDs è il risultato storico della V1, non un numero forzato dal codice. La run ricostruita va eseguita e poi valutata nel notebook 04.

In [1]:
from pathlib import Path
import sys, gc
import cv2
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from scipy.optimize import linear_sum_assignment

DRIVE_ROOT = Path(
    "/Users/marcodalbis/Library/CloudStorage/"
    "GoogleDrive-marco.dalbis@gmail.com/"
    "Il mio Drive/"
    "basketball-thesis"
)

SEQUENCE_NAME = "v_00HRwkvvjtQ_c001"

FRAMES_DIR = DRIVE_ROOT / "datasets" / "sportsmot" / "val" / SEQUENCE_NAME / "img1"
CHECKPOINT_PATH = DRIVE_ROOT / "checkpoints" / "rfdetr_nano_10epochs" / "checkpoint_best_total.pth"

SAM2_REPO = Path("/Users/marcodalbis/Desktop/Tesi Magistrale/sam2-official")
SAM2_CHECKPOINT = SAM2_REPO / "checkpoints" / "sam2.1_hiera_small.pt"
SAM2_CONFIG = "configs/sam2.1/sam2.1_hiera_s.yaml"

OUTPUT_DIR = DRIVE_ROOT / "experiments" / "sam2_tracking"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRACKS_CSV = OUTPUT_DIR / f"{SEQUENCE_NAME}_tracks_RECONSTRUCTED.csv"

CHUNK_SIZE = 100
OVERLAP = 10
STEP = CHUNK_SIZE - OVERLAP
DETECTION_THRESHOLD = 0.30
MATCH_IOU_THRESHOLD = 0.30

frame_paths = sorted(FRAMES_DIR.glob("*.jpg"))
N_FRAMES = len(frame_paths)

print("Frames:", N_FRAMES)
print("Frames dir:", FRAMES_DIR)
print("RF-DETR checkpoint:", CHECKPOINT_PATH, CHECKPOINT_PATH.exists())
print("SAM2 repo:", SAM2_REPO, SAM2_REPO.exists())
print("Output:", TRACKS_CSV)

assert N_FRAMES > 0
assert CHECKPOINT_PATH.exists()
assert SAM2_REPO.exists()


Frames: 1165
Frames dir: /Users/marcodalbis/Library/CloudStorage/GoogleDrive-marco.dalbis@gmail.com/Il mio Drive/basketball-thesis/datasets/sportsmot/val/v_00HRwkvvjtQ_c001/img1
RF-DETR checkpoint: /Users/marcodalbis/Library/CloudStorage/GoogleDrive-marco.dalbis@gmail.com/Il mio Drive/basketball-thesis/checkpoints/rfdetr_nano_10epochs/checkpoint_best_total.pth True
SAM2 repo: /Users/marcodalbis/Desktop/Tesi Magistrale/sam2-official True
Output: /Users/marcodalbis/Library/CloudStorage/GoogleDrive-marco.dalbis@gmail.com/Il mio Drive/basketball-thesis/experiments/sam2_tracking/v_00HRwkvvjtQ_c001_tracks_RECONSTRUCTED.csv


In [2]:
# RF-DETR
from rfdetr import RFDETRNano

detector = RFDETRNano(pretrain_weights=str(CHECKPOINT_PATH))
print("RF-DETR loaded")


/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/.venv/lib/python3.11/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
[2026-10-08 13:01:48] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-10-08 13:01:48] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-10-08 13:01:49] [WARNING] rf-detr - Checkpoint has 1 classes but model is configured for 90. Using checkpoint class count (1). Pass num_classes=1 to suppress this warning.


RF-DETR loaded


In [3]:
# SAM2
if str(SAM2_REPO) not in sys.path:
    sys.path.insert(0, str(SAM2_REPO))

from sam2.build_sam import build_sam2_video_predictor

sam2_predictor = build_sam2_video_predictor(
    SAM2_CONFIG,
    str(SAM2_CHECKPOINT),
    device="mps" if torch.backends.mps.is_available() else "cpu"
)

print("SAM2 loaded")


SAM2 loaded


In [4]:
def detect_players(frame_number, threshold=DETECTION_THRESHOLD):
    frame_path = FRAMES_DIR / f"{frame_number:06d}.jpg"
    bgr = cv2.imread(str(frame_path))
    if bgr is None:
        raise FileNotFoundError(frame_path)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    det = detector.predict(rgb, threshold=threshold)
    boxes = [np.asarray(b, dtype=np.float32) for b in det.xyxy]
    confs = [float(c) for c in det.confidence]
    return boxes, confs

def bbox_iou(a, b):
    x1=max(a[0],b[0]); y1=max(a[1],b[1])
    x2=min(a[2],b[2]); y2=min(a[3],b[3])
    iw=max(0.0,x2-x1); ih=max(0.0,y2-y1)
    inter=iw*ih
    aa=max(0,a[2]-a[0])*max(0,a[3]-a[1])
    ab=max(0,b[2]-b[0])*max(0,b[3]-b[1])
    return inter/(aa+ab-inter+1e-9)

def mask_to_box(mask):
    m = np.asarray(mask)
    while m.ndim > 2:
        m = m[0]
    ys, xs = np.where(m > 0)
    if len(xs) == 0:
        return None
    return np.array([xs.min(), ys.min(), xs.max()+1, ys.max()+1], dtype=np.float32)

def match_boxes(old_items, new_boxes, threshold=MATCH_IOU_THRESHOLD):
    # old_items = [(global_id, box), ...]
    if not old_items or not new_boxes:
        return {}, set(range(len(new_boxes)))

    mat = np.zeros((len(old_items), len(new_boxes)), dtype=np.float32)
    for i, (_, ob) in enumerate(old_items):
        for j, nb in enumerate(new_boxes):
            mat[i,j] = bbox_iou(ob, nb)

    rr, cc = linear_sum_assignment(-mat)
    matches = {}
    used_new = set()
    for i,j in zip(rr,cc):
        if mat[i,j] >= threshold:
            matches[j] = old_items[i][0]
            used_new.add(j)
    return matches, set(range(len(new_boxes))) - used_new


In [5]:
# Smoke test RF-DETR
boxes, confs = detect_players(1)
print("Frame 1 detections:", len(boxes))


[2026-10-08 13:02:01] [WARNING] rf-detr - Model is not optimized for inference. Latency may be higher than expected. For full GPU throughput (e.g. ~8x on T4 via FP16 Tensor Cores), call model.inference(dtype=torch.float16).


Frame 1 detections: 8


Tracking V1 ricostruito
Ogni chunk contiene 100 frame. I chunk successivi ripartono 10 frame prima.
Sul primo frame del chunk:
1. RF-DETR rileva i giocatori;
2. le detection vengono associate alle tracce provenienti dal chunk precedente tramite IoU;
3. solo le detection non associate ricevono un nuovo ID globale;
4. SAM2 propaga gli oggetti nel chunk.
Le osservazioni dell'overlap vengono salvate una sola volta.

In [ ]:
all_rows = []
next_global_id = 0
previous_continuity = []   # [(global_id, box), ...]

chunk_starts = list(range(1, N_FRAMES + 1, STEP))

for chunk_idx, start_frame in enumerate(chunk_starts, start=1):
    end_frame = min(start_frame + CHUNK_SIZE - 1, N_FRAMES)
    chunk_len = end_frame - start_frame + 1

    print("\n" + "="*70)
    print(f"CHUNK {chunk_idx}/{len(chunk_starts)}: {start_frame} -> {end_frame}")
    print("="*70)

    # cartella temporanea di JPEG per SAM2
    chunk_dir = OUTPUT_DIR / "_sam2_chunk_tmp"
    chunk_dir.mkdir(parents=True, exist_ok=True)

    # pulizia dei vecchi jpg
    for p in chunk_dir.glob("*.jpg"):
        p.unlink()

    for local_idx, global_frame in enumerate(range(start_frame, end_frame+1)):
        src = FRAMES_DIR / f"{global_frame:06d}.jpg"
        dst = chunk_dir / f"{local_idx:06d}.jpg"
        # SAM2 richiede una sequenza numerata; copiamo senza ricodifica
        dst.write_bytes(src.read_bytes())

    boxes, confs = detect_players(start_frame)
    print("RF-DETR detections:", len(boxes))

    detection_to_id = {}
    unmatched = set(range(len(boxes)))

    if previous_continuity:
        detection_to_id, unmatched = match_boxes(previous_continuity, boxes)
        print("Inherited IDs:", sorted(set(detection_to_id.values())))
    else:
        print("Initial chunk")

    new_ids = []
    for j in sorted(unmatched):
        detection_to_id[j] = next_global_id
        new_ids.append(next_global_id)
        next_global_id += 1

    print("New IDs from RF-DETR:", new_ids)
    print("Seed IDs:", sorted(detection_to_id.values()))

    state = sam2_predictor.init_state(video_path=str(chunk_dir))

    for det_idx, box in enumerate(boxes):
        sam2_predictor.add_new_points_or_box(
            inference_state=state,
            frame_idx=0,
            obj_id=int(detection_to_id[det_idx]),
            box=np.asarray(box, dtype=np.float32)
        )

    chunk_rows = []
    frame_boxes_by_id = {}

    for out_frame_idx, out_obj_ids, out_mask_logits in sam2_predictor.propagate_in_video(state):
        global_frame = start_frame + int(out_frame_idx)
        if global_frame > end_frame:
            continue

        frame_boxes_by_id[global_frame] = {}

        for k, obj_id in enumerate(out_obj_ids):
            mask = (out_mask_logits[k] > 0).detach().cpu().numpy()
            box = mask_to_box(mask)
            if box is None:
                continue

            gid = int(obj_id)
            frame_boxes_by_id[global_frame][gid] = box

            # non duplicare i primi OVERLAP frame dei chunk > 1
            if chunk_idx > 1 and global_frame < start_frame + OVERLAP:
                continue

            chunk_rows.append({
                "frame": global_frame,
                "track_id": gid,
                "x1": float(box[0]),
                "y1": float(box[1]),
                "x2": float(box[2]),
                "y2": float(box[3]),
                "confidence": 1.0,
                "class_id": 0,
            })

    all_rows.extend(chunk_rows)

    # Continuità: frame che diventerà l'inizio del chunk successivo
    next_start = start_frame + STEP
    if next_start <= N_FRAMES:
        continuity_frame = next_start
        fb = frame_boxes_by_id.get(continuity_frame, {})
        previous_continuity = [(gid, box) for gid, box in fb.items()]
        print("Continuity frame:", continuity_frame)
        print("Inherited for next chunk:", sorted(fb.keys()))
    else:
        previous_continuity = []

    df_partial = pd.DataFrame(all_rows)
    df_partial.to_csv(TRACKS_CSV, index=False)

    print("Saved observations:", len(df_partial))
    print("Unique IDs:", df_partial["track_id"].nunique() if len(df_partial) else 0)
    print("Last saved frame:", int(df_partial["frame"].max()) if len(df_partial) else None)

    try:
        sam2_predictor.reset_state(state)
    except Exception:
        pass
    del state
    gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()



CHUNK 1/13: 1 -> 100
RF-DETR detections: 8
Initial chunk
New IDs from RF-DETR: [0, 1, 2, 3, 4, 5, 6, 7]
Seed IDs: [0, 1, 2, 3, 4, 5, 6, 7]


frame loading (JPEG): 100%|██████████| 100/100 [00:02<00:00, 43.16it/s]
/Users/marcodalbis/Desktop/Tesi Magistrale/sam2-official/sam2/sam2_video_predictor.py:786: UserWarning: cannot import name '_C' from 'sam2' (/Users/marcodalbis/Desktop/Tesi Magistrale/sam2-official/sam2/__init__.py)

Skipping the post-processing step due to the error above. You can still use SAM 2 and it's OK to ignore the error above, although some post-processing functionality may be limited (which doesn't affect the results in most cases; see https://github.com/facebookresearch/sam2/blob/main/INSTALL.md).
  pred_masks_gpu = fill_holes_in_mask_scores(
propagate in video: 100%|██████████| 100/100 [13:57<00:00,  8.37s/it]


Continuity frame: 91
Inherited for next chunk: [0, 1, 3, 4, 5, 6, 7]
Saved observations: 781
Unique IDs: 8
Last saved frame: 100

CHUNK 2/13: 91 -> 190
RF-DETR detections: 8
Inherited IDs: [0, 1, 3, 4, 5, 7]
New IDs from RF-DETR: [8, 9]
Seed IDs: [0, 1, 3, 4, 5, 7, 8, 9]


frame loading (JPEG): 100%|██████████| 100/100 [00:03<00:00, 31.52it/s]
/Users/marcodalbis/Desktop/Tesi Magistrale/sam2-official/sam2/sam2_video_predictor.py:786: UserWarning: cannot import name '_C' from 'sam2' (/Users/marcodalbis/Desktop/Tesi Magistrale/sam2-official/sam2/__init__.py)

Skipping the post-processing step due to the error above. You can still use SAM 2 and it's OK to ignore the error above, although some post-processing functionality may be limited (which doesn't affect the results in most cases; see https://github.com/facebookresearch/sam2/blob/main/INSTALL.md).
  pred_masks_gpu = fill_holes_in_mask_scores(
propagate in video: 100%|██████████| 100/100 [26:55<00:00, 16.15s/it]  


Continuity frame: 181
Inherited for next chunk: [0, 1, 3, 4, 5, 7, 8, 9]
Saved observations: 1501
Unique IDs: 10
Last saved frame: 190

CHUNK 3/13: 181 -> 280
RF-DETR detections: 10
Inherited IDs: [0, 1, 3, 4, 7, 8]
New IDs from RF-DETR: [10, 11, 12, 13]
Seed IDs: [0, 1, 3, 4, 7, 8, 10, 11, 12, 13]


frame loading (JPEG): 100%|██████████| 100/100 [00:02<00:00, 41.94it/s]
/Users/marcodalbis/Desktop/Tesi Magistrale/sam2-official/sam2/sam2_video_predictor.py:786: UserWarning: cannot import name '_C' from 'sam2' (/Users/marcodalbis/Desktop/Tesi Magistrale/sam2-official/sam2/__init__.py)

Skipping the post-processing step due to the error above. You can still use SAM 2 and it's OK to ignore the error above, although some post-processing functionality may be limited (which doesn't affect the results in most cases; see https://github.com/facebookresearch/sam2/blob/main/INSTALL.md).
  pred_masks_gpu = fill_holes_in_mask_scores(
propagate in video:  91%|█████████ | 91/100 [29:22<02:07, 14.13s/it]  

In [ ]:
tracks_df = pd.DataFrame(all_rows).sort_values(["frame","track_id"]).reset_index(drop=True)
tracks_df.to_csv(TRACKS_CSV, index=False)

print("Saved:", TRACKS_CSV)
print("Observations:", len(tracks_df))
print("Frames:", tracks_df["frame"].min(), "->", tracks_df["frame"].max())
print("Unique IDs:", tracks_df["track_id"].nunique())
print("IDs:", sorted(tracks_df["track_id"].unique().tolist()))


In [ ]:
# Video di controllo
VIDEO_OUTPUT = OUTPUT_DIR / f"{SEQUENCE_NAME}_tracking_v1_RECONSTRUCTED.mp4"

first = cv2.imread(str(FRAMES_DIR / "000001.jpg"))
h, w = first.shape[:2]
writer = cv2.VideoWriter(str(VIDEO_OUTPUT), cv2.VideoWriter_fourcc(*"mp4v"), 25, (w,h))

for frame_number in tqdm(range(1, N_FRAMES+1), desc="Creating video"):
    frame = cv2.imread(str(FRAMES_DIR / f"{frame_number:06d}.jpg"))
    if frame is None:
        continue
    rows = tracks_df[tracks_df.frame == frame_number]
    for _, r in rows.iterrows():
        x1,y1,x2,y2 = map(int,[r.x1,r.y1,r.x2,r.y2])
        tid=int(r.track_id)
        cv2.rectangle(frame,(x1,y1),(x2,y2),(255,255,255),2)
        cv2.putText(frame,f"ID {tid}",(x1,max(20,y1-8)),
                    cv2.FONT_HERSHEY_SIMPLEX,0.6,(255,255,255),2,cv2.LINE_AA)
    writer.write(frame)

writer.release()
print("Video:", VIDEO_OUTPUT)
print("IDs:", tracks_df.track_id.nunique())
